In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1779942710042_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
spark = (
    SparkSession.builder
    .appName("ShopStreamAnalytics")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
df = (
    spark.read
    .option("multiLine", "false")
    .json("s3://shopstream-001/data/")
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- action: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- element_id: string (nullable = true)
 |-- element_type: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page_type: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- query: string (nullable = true)
 |-- referrer: string (nullable = true)
 |-- results_count: long (nullable = true)
 |-- session_id: long (nullable = true)
 |-- time_on_page_seconds: long (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- x_position: long (nullable = true)
 |-- y_position: long (nullable = true)
 |-- year: integer (nullable = true)

In [5]:
df.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+--------+-------------+-----------+----------+------------+----------+---------+----------------------------------------+-----+----------+------------+-----+--------+-------------+----------+--------------------+-------------------+-----------------+----------+----------+----+
|action|category|country      |device_type|element_id|element_type|event_type|page_type|page_url                                |price|product_id|product_name|query|referrer|results_count|session_id|time_on_page_seconds|timestamp          |user_id          |x_position|y_position|year|
+------+--------+-------------+-----------+----------+------------+----------+---------+----------------------------------------+-----+----------+------------+-----+--------+-------------+----------+--------------------+-------------------+-----------------+----------+----------+----+
|null  |null    |French Guiana|tablet     |null      |null        |page_view |home     |https://shopstream.com/home             |null |null   

In [6]:
df.groupBy("event_type").count().show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+------+
|  event_type| count|
+------------+------+
|   page_view|252318|
|product_view| 92471|
|       click| 46207|
|  cart_event| 46482|
|      search| 69164|
+------------+------+

In [44]:
#limpieza de datos

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
df = df.dropDuplicates()

In [ ]:
df = df.withColumn("event_date", to_date(col("timestamp")))

In [ ]:
# =========================================================
# FILTRAR EVENTOS POR TIPO
# =========================================================

from pyspark.sql.functions import *

page_views = df.filter(
    col("event_type") == "page_view"
)

clicks = df.filter(
    col("event_type") == "click"
)

searches = df.filter(
    col("event_type") == "search"
)

product_views = df.filter(
    col("event_type") == "product_view"
)

cart_events = df.filter(
    col("event_type") == "cart_event"
)

In [42]:
# =========================================================
# PAGE_VIEW CLEANING
# =========================================================

# eliminar registros sin campos críticos
page_views = page_views.dropna(
    subset=[
        "page_url",
        "page_type",
        "timestamp"
    ]
)

# imputación válida:


avg_time = page_views.select(
    avg("time_on_page_seconds")
).first()[0]

page_views = page_views.fillna({
    "time_on_page_seconds": avg_time
})

# imputación categórica válida
page_views = page_views.fillna({
    "country": "UNKNOWN",
    "device_type": "UNKNOWN",
    "referrer": "DIRECT"
})

# normalización de texto
page_views = (
    page_views
    .withColumn(
        "page_type",
        lower(trim(col("page_type")))
    )
    .withColumn(
        "device_type",
        lower(trim(col("device_type")))
    )
)

# tipos correctos
page_views = (
    page_views
    .withColumn(
        "time_on_page_seconds",
        col("time_on_page_seconds").cast("double")
    )
    .withColumn(
        "timestamp",
        to_timestamp("timestamp")
    )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [43]:
# =========================================================
# PRODUCT_VIEW CLEANING
# =========================================================

product_views = product_views.dropna(
    subset=[
        "product_id",
        "category",
        "price"
    ]
)

# normalización
product_views = (
    product_views
    .withColumn(
        "category",
        lower(trim(col("category")))
    )
)

# tipos
product_views = (
    product_views
    .withColumn(
        "price",
        col("price").cast("double")
    )
    .withColumn(
        "time_on_page_seconds",
        col("time_on_page_seconds").cast("double")
    )
    .withColumn(
        "timestamp",
        to_timestamp("timestamp")
    )
)

# imputación válida
avg_product_time = product_views.select(
    avg("time_on_page_seconds")
).first()[0]

product_views = product_views.fillna({
    "time_on_page_seconds": avg_product_time
})

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Interrupted by user


In [ ]:
# =========================================================
# CLICK CLEANING
# =========================================================

clicks = clicks.dropna(
    subset=[
        "x_position",
        "y_position",
        "element_id"
    ]
)

# tipos
clicks = (
    clicks
    .withColumn(
        "x_position",
        col("x_position").cast("int")
    )
    .withColumn(
        "y_position",
        col("y_position").cast("int")
    )
    .withColumn(
        "timestamp",
        to_timestamp("timestamp")
    )
)

# normalización
clicks = clicks.withColumn(
    "element_type",
    lower(trim(col("element_type")))
)

In [ ]:
# =========================================================
# SEARCH CLEANING
# =========================================================

searches = searches.dropna(
    subset=[
        "query"
    ]
)

# normalización
searches = searches.withColumn(
    "query",
    lower(trim(col("query")))
)

# tipos
searches = (
    searches
    .withColumn(
        "results_count",
        col("results_count").cast("int")
    )
    .withColumn(
        "timestamp",
        to_timestamp("timestamp")
    )
)

In [ ]:
# =========================================================
# CART_EVENT CLEANING
# =========================================================

cart_events = cart_events.dropna(
    subset=[
        "product_id",
        "action"
    ]
)

# normalización
cart_events = (
    cart_events
    .withColumn(
        "action",
        lower(trim(col("action")))
    )
)

# tipos
cart_events = cart_events.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

In [ ]:
# =========================================================
# REMOVE DUPLICATES
# =========================================================

page_views = page_views.dropDuplicates()

clicks = clicks.dropDuplicates()

searches = searches.dropDuplicates()

product_views = product_views.dropDuplicates()

cart_events = cart_events.dropDuplicates()

In [ ]:
# Normalizacion

In [ ]:
# =========================================================
# TEXT NORMALIZATION
# =========================================================

from pyspark.sql.functions import *

# page_views
page_views = (
    page_views
    .withColumn(
        "page_type",
        lower(trim(col("page_type")))
    )
    .withColumn(
        "device_type",
        lower(trim(col("device_type")))
    )
    .withColumn(
        "country",
        upper(trim(col("country")))
    )
    .withColumn(
        "referrer",
        lower(trim(col("referrer")))
    )
)

# product_views
product_views = (
    product_views
    .withColumn(
        "category",
        lower(trim(col("category")))
    )
)

# clicks
clicks = (
    clicks
    .withColumn(
        "element_type",
        lower(trim(col("element_type")))
    )
)

# cart_events
cart_events = (
    cart_events
    .withColumn(
        "action",
        lower(trim(col("action")))
    )
)

# searches
searches = (
    searches
    .withColumn(
        "query",
        lower(trim(col("query")))
    )
)

In [ ]:
# =========================================================
# MIN-MAX NORMALIZATION FOR PRICE
# =========================================================

price_stats = product_views.select(
    min("price").alias("min_price"),
    max("price").alias("max_price")
).first()

min_price = price_stats["min_price"]
max_price = price_stats["max_price"]

product_views = product_views.withColumn(
    "price_norm",
    (
        col("price") - min_price
    ) / (
        max_price - min_price
    )
)

In [ ]:
# =========================================================
# Z-SCORE NORMALIZATION
# =========================================================

stats = page_views.select(
    avg("time_on_page_seconds").alias("mean"),
    stddev("time_on_page_seconds").alias("std")
).first()

mean_time = stats["mean"]
std_time = stats["std"]

page_views = page_views.withColumn(
    "time_zscore",
    (
        col("time_on_page_seconds") - mean_time
    ) / std_time
)

In [ ]:
# =========================================================
# 1: TOP 20 PÁGINAS POR TIEMPO PROMEDIO
# =========================================================

top_pages = (
    page_views
    .groupBy("page_url")
    .agg(
        avg("time_on_page_seconds").alias("avg_time")
    )
    .orderBy(desc("avg_time"))
    .limit(20)
)

top_pages.show()

In [ ]:
# =========================================================
# 2: BOUNCE RATE POR PAGE_TYPE
# =========================================================

session_views = (
    page_views
    .groupBy("session_id", "page_type")
    .agg(count("*").alias("pv_count"))
)

bounce_rate = (
    session_views
    .groupBy("page_type")
    .agg(
        (
            sum(when(col("pv_count") == 1, 1).otherwise(0)) /
            count("*")
        ).alias("bounce_rate")
    )
)

bounce_rate.show()

In [ ]:
# =========================================================
# 3: FUNNEL DE CONVERSIÓN
# =========================================================

page_users = page_views.select("user_id").distinct()
product_users = product_views.select("user_id").distinct()
cart_users = cart_events.select("user_id").distinct()

funnel_df = spark.createDataFrame([
    ("page_view", page_users.count()),
    ("product_view", product_users.count()),
    ("cart_event", cart_users.count())
], ["stage", "users"])

funnel_df.show()

In [ ]:
# =========================================================
#  4: PRODUCTOS: VISTOS VS CARRITO
# =========================================================

views = (
    product_views
    .groupBy("product_name")
    .agg(count("*").alias("views"))
)

carts = (
    cart_events
    .filter(col("action") == "add")
    .groupBy("product_name")
    .agg(count("*").alias("adds"))
)

product_perf = (
    views.join(carts, "product_name", "left")
    .fillna(0)
    .withColumn("gap", col("views") - col("adds"))
    .orderBy(desc("gap"))
)

product_perf.show()

In [ ]:
# =========================================================
# 5: NAVIGATION PATHS
# =========================================================

from pyspark.sql.window import Window

w = Window.partitionBy("session_id").orderBy("timestamp")

ordered = page_views.withColumn(
    "page",
    col("page_url")
).withColumn(
    "rn",
    row_number().over(w)
)

paths = (
    ordered
    .groupBy("session_id")
    .agg(
        collect_list("page").alias("path")
    )
)

top_paths = (
    paths
    .groupBy("path")
    .count()
    .orderBy(desc("count"))
    .limit(10)
)

top_paths.show(truncate=False)

In [ ]:
# =========================================================
#  6: ENGAGEMENT POR DISPOSITIVO Y PAÍS
# =========================================================

device_country = (
    page_views
    .groupBy("device_type", "country")
    .agg(
        avg("time_on_page_seconds").alias("avg_time")
    )
    .orderBy(desc("avg_time"))
)

device_country.show()

In [ ]:
def write_to_s3(metrics_dict, base_path="s3://shopstream-001/analytics/"):

    for metric_name, df in metrics_dict.items():

        print(f"[EXPORTING] {metric_name}")

        # -----------------------------
        # asegurar columna de partición
        # -----------------------------
        if "event_date" not in df.columns:
            df = df.withColumn("event_date", to_date(col("timestamp")))

        # -----------------------------
        # write Parquet partitioned
        # -----------------------------
        (
            df.write
            .mode("overwrite")   # en producción podrías usar append
            .partitionBy("event_date")
            .parquet(base_path + metric_name + "/")
        )

        print(f"[DONE] {metric_name} -> {base_path}{metric_name}/")

In [ ]:
metrics = {
    "top_pages": top_pages,
    "bounce_rate": bounce_rate,
    "funnel": funnel_df,
    "product_perf": product_perf,
    "top_paths": top_paths,
    "device_country": device_country,
}

In [ ]:

write_to_s3(metrics)